# Module 4 — Compression and release decisions


This is the complete teaching lab for Module 4. Run setup first, then follow the cells in order. Section headings are learning topics; [the current lesson map](../../docs/LESSON_MAP.md) maps them to the course videos. Collapsing a section does not skip execution.

**Practice versus assessment.** Worked examples teach the method; they are not completed capstone evidence. Use the [RiskLens capstone repository](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone) for exact evidence thresholds and all 24 Met / Not Yet criteria. Synthetic examples, tiny practice splits and unreviewed judge outputs do not establish mastery.

**Runtime.** Read [Colab setup and artifact handoffs](../../docs/COLAB.md). The CPU exercises run separately in `practice.ipynb` and `solutions.ipynb`. This lab uses model downloads and GPU execution. Modules 3–5 require your saved artifacts and explicit train/load choices. A notebook that stops for missing artifacts is not a completed run. See [what was tested](../../docs/VALIDATION.md).


## Shared setup before running

---
### 0 · Setup

In [ ]:
"""Run this cell first in a fresh Colab session, before importing model libraries."""
import importlib.metadata as _metadata
import platform as _platform
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
from pathlib import Path as _Path

COURSE_PACKAGES = [
    "transformers==5.22.0", "trl==1.17.0", "peft==0.21.2",
    "accelerate==1.15.0", "datasets==5.1.0", "sentence-transformers==6.2.0",
    "faiss-cpu==1.15.1", "num2words==0.5.14",
    "pillow==13.0.0", "scikit-learn==1.9.1", "matplotlib==3.11.2", "requests==2.34.2",
]
if _sys.version_info < (3, 11):
    raise RuntimeError("Use a current Colab Python 3.11+ runtime.")
try:
    _torch_version = _metadata.version("torch")
except _metadata.PackageNotFoundError as _error:
    raise RuntimeError("Use Colab's preinstalled matched PyTorch/torchvision runtime; see docs/COLAB.md.") from _error
_constraints = ["torch==" + _torch_version]
try:
    _constraints.append("torchvision==" + _metadata.version("torchvision"))
except _metadata.PackageNotFoundError:
    raise RuntimeError("The vision labs need a matching torchvision installation; use a fresh Colab runtime.")
if _platform.system() == "Linux":
    COURSE_PACKAGES.append("bitsandbytes==0.50.2")
_before = {}
for _name in ["transformers", "trl", "peft", "datasets", "sentence_transformers", "numpy"]:
    if _name in _sys.modules:
        _before[_name] = getattr(_sys.modules[_name], "__version__", None)
with _tempfile.TemporaryDirectory() as _temporary:
    _constraint_file = _Path(_temporary) / "preserve-colab-torch.txt"
    _constraint_file.write_text("\n".join(_constraints))
    _subprocess.run([_sys.executable, "-m", "pip", "install", "--quiet", "--upgrade-strategy", "only-if-needed",
                     "-c", str(_constraint_file), *COURSE_PACKAGES], check=True)
if _before:
    raise RuntimeError("Packages installed. Restart the Colab session once, then run from the setup cell again before importing models.")
print("Course stack installed; existing PyTorch/torchvision CUDA wheels preserved.")
print("If installation failed, stop here. Start a fresh GPU runtime and inspect the resolver message.")


In [ ]:
import torch, time, json, os

assert torch.cuda.is_available(), "This session needs a GPU."
print("Device:", torch.cuda.get_device_name(0))

BF16_OK = torch.cuda.is_bf16_supported()          # False on T4 (pre-Ampere)
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print("Using dtype:", DTYPE)

### Shared artifact setup
Restore the **actual Module 3 adapter folder**, including its tokenizer, course_manifest.json and frozen_data_split.json. Set ADAPTER_DIR below. This notebook does not silently train a replacement Analyst. The immutable Week 4 source retains the original synthetic bootstrap for historical reference. A practice-mode Module 3 artifact remains practice, even when loaded correctly.

Use a fresh GPU runtime for measurements. Record installed versions after a successful rehearsal; broad dependency ranges are not a reproducibility lock. The original Module 3 test rows may already have been inspected. They support a paired regression check; do not claim a new untouched final test after choosing configurations from their results.


In [ ]:
from pathlib import Path
from importlib.metadata import version
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
import hashlib, json, gc, math

ADAPTER_DIR = Path("analyst-lora-adapter")
FOLDED_DIR = Path("analyst-merged")
FOLD_MODE = "load"  # Choose create deliberately once; preserve existing artifacts.

def stable_sha(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""): h.update(block)
    return h.hexdigest()

def artifact_files(folder):
    files = sorted(x for x in Path(folder).rglob("*") if x.is_file() and x.name != "reload_check.json")
    if not files: raise FileNotFoundError(f"No artifact files in {folder}")
    return {str(x.relative_to(folder)): file_sha(x) for x in files}

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str))

manifest_path = ADAPTER_DIR / "course_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Restore the actual Module3 adapter, manifest, tokenizer and frozen split first")
artifact_manifest = json.loads(manifest_path.read_text())
split_record = json.loads((ADAPTER_DIR / "frozen_data_split.json").read_text())
if stable_sha(split_record) != artifact_manifest["split_sha256"]:
    raise ValueError("Adapter manifest and frozen split do not match")
if artifact_manifest["compute_dtype"] != str(DTYPE):
    raise ValueError("Match the recorded Module3 compute dtype on compatible hardware before this comparison")
BASE_ID = artifact_manifest["base_id"]
BASE_REVISION = artifact_manifest["resolved_base_revision"]
if not BASE_REVISION or BASE_REVISION == "main":
    raise ValueError("Use the resolved base commit recorded in Module3")
test_rows = split_record["splits"]["test"]
if not test_rows or len({x["row_id"] for x in test_rows}) != len(test_rows):
    raise ValueError("Nonempty unique frozen row IDs are required")
for row in test_rows:
    if not all(isinstance(row.get(k),str) and row[k].strip() for k in ["input","output"]):
        raise ValueError("Every frozen row requires complete input and reference output")
ADAPTER_FILES = artifact_files(ADAPTER_DIR)
tok = AutoTokenizer.from_pretrained(str(ADAPTER_DIR)); tok.padding_side = "right"
if tok.pad_token is None: tok.pad_token = tok.eos_token

def make_input(defect_note, evidence):
    return f"Defect note from visual inspection:\n{defect_note}\n\nRetrieved evidence:\n{evidence}\n\nWrite a root cause analysis report."

print("Module3 provenance", BASE_ID, BASE_REVISION, artifact_manifest["data_mode"])
print("Frozen regression rows", len(test_rows), "split hash", stable_sha(split_record))
print("Runtime", {x: version(x) for x in ["torch","transformers","peft","bitsandbytes"]})


In [ ]:
# No automatic bootstrap training. Missing artifacts stop the notebook above.
print("Loaded artifact metadata only; no model has been trained or evaluated by this cell.")


### Preserve the original reference
The original Module 3 reference uses its recorded NF4 base preparation plus adapter. Folding the adapter into a half-precision base changes that representation. Keep three labels: `module3_reference`, `folded_half`, and `folded_nf4`. Compare both transformed candidates with the original reference before a project replacement. The half-versus-NF4 pair isolates the load-time quantization of the folded checkpoint.


## Topic — Data Parallelism & Model Parallelism Explained

**Theory · 8 minutes**

Compare what each device stores, computes and communicates under data parallelism, tensor parallelism, pipeline parallelism and training-state sharding. All examples are illustrative; no multi-GPU runtime or distributed-training launch is needed for this video.


### Data and model parallelism concept companion

**Data parallelism:** matching replicas process disjoint portions of a batch, compute local gradients and synchronize them before consistent optimizer updates. Ordinary replicas retain complete parameter and optimizer state; splitting examples does not split those weights. Two 8 GB devices do not fit 12 GB of weights per replica simply because aggregate capacity is 16 GB. Smaller local batches may reduce activation memory for the same combined batch.

**Exact scalar toy:** all examples share `w=10`; losses are `L_j(w)=0.5*(w-a_j)**2` with targets `[8,6,4,2]`. Individual gradients are `[2,4,6,8]`. Two local batches of two examples give means 3 and 7; averaging gives 5, equal to the four-example mean. With plain SGD and learning rate 0.1, both weights become 9.5. The actual course trains adapters with AdamW, not this scalar optimizer. Matching initial states, optimizer behavior, normalization, stochastic operations and numerical precision affect equivalence.

**Unequal local means:** repartition the same examples into one gradient `[2]` and three gradients `[4,6,8]`. Local means are 2 and 6. Their simple average is 4, which is wrong for the intended four-example mean. Count-weighted combination `(1*2+3*6)/4` restores 5. For token-mean language-model losses, use the appropriate valid supervised-token counts; example counts alone may not match the loss definition. Inspect the trainer's reduction and distributed normalization.

The all-reduce icon is a logical collective, not a required central server. DDP does not automatically shard a dataset merely because the model is wrapped; the data input must be distributed appropriately. Do not treat two different local updates as synchronous data-parallel training.

**Tensor parallelism:** partition one operation and communicate the tensors its layout requires. For `W=[[1,2],[3,4]]` and `x=[1,2]`, output-row partitions yield 5 and 11; concatenation gives `[5,11]`, the original `W x`. Input-dimension partitions would instead sum partial outputs. Nonlinearities and subsequent operations constrain valid communication locations. This example uses a column-vector convention and calls the split by its literal output rows; library layer-layout names may follow transposed storage conventions.

**Pipeline parallelism:** successive stages on different devices exchange forward activations and backward activation gradients. One stage cannot finish independently of those tensors. Microbatches overlap different examples across stages. The animation's balanced two-stage, three-microbatch forward schedule occupies six of eight stage-slots: 75% utilization. It excludes backward work and communication overhead. Real training schedules have their own bubbles, balance and activation-memory trade-offs.

**Training-state sharding:** full sharding distributes persistent parameters, gradients and optimizer states across data-parallel workers, gathering required parameter units and reducing gradient pieces around computation. Illustrative full-training ledger `P=8 GB, G=8 GB, O=16 GB` totals 32 GB per ordinary replica or an ideal 16 GB persistent share on each of two devices. Activations, gathered units, temporary buffers and allocator overhead add to peak memory. The ledger is not a measurement of QLoRA. Sharding persistent state is not the same as assigning sequential layers to pipeline stages.

**Timing:** an illustrative fixed-global-batch step takes 180 ms on one device, versus 80 ms local compute plus 20 ms exposed synchronization on two, giving 1.8× speedup. Actual communication may overlap compute; topology and load balance matter. Do not infer single-request inference latency from training throughput or installed device count. Parallel strategies can combine across separate device-group dimensions.

**Source boundary:** the inherited notebook's `device_map="auto"` loads/places model components. Big Model Inference placement is not a DDP launch, training-data partition, gradient reduction or optimizer schedule. No distributed run has been executed in this recording notebook. This theory section can be read without running the shared GPU setup; tutorial sections require their actual prerequisites.

Primary references reference snapshot: 21 October 2026: [PyTorch DDP](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html), [Tensor Parallel tutorial](https://docs.pytorch.org/tutorials/intermediate/TP_tutorial.html), [FSDP tutorial](https://docs.pytorch.org/tutorials/intermediate/FSDP_tutorial.html), [GPipe](https://arxiv.org/abs/1811.06965), [ZeRO](https://arxiv.org/abs/1910.02054), [Big Model Inference](https://huggingface.co/docs/accelerate/usage_guides/big_modeling).


## Topic — Optimizing Training & Inference

**Theory · 8 minutes · resource trade-offs**

Choose a mechanism by the actual bottleneck. Mixed precision changes operation precision and tensor representation; smaller microbatches plus gradient accumulation reduce simultaneous activation demand while delaying the optimizer update; activation checkpointing recomputes intermediates during backward computation. A model checkpoint saved to disk is a different use of the word checkpoint.

During causal autoregressive inference, a KV cache retains earlier keys and values, while each new query still reads the retained history. Batching can improve aggregate throughput while increasing individual completion latency. Record output lengths, batch size, warmup, device/runtime and timing scope before comparing.

**Illustrative arithmetic, not measured notebook results**

- Selected training components: 8 GB persistent + 4 GB saved activations = 12 GB. A toy checkpoint policy reduces saved activations to 2 GB, leaving 10 GB **before workspace and allocator overhead**; this does not prove a real 10 GB device can run it.
- One 500-million-element buffer: FP32 2 GB; sixteen-bit storage 1 GB. With 8 GB of unchanged other state, the accounted total is 10 → 9 GB, not half. AMP does not force all state into one precision.
- Four equal two-example microbatches: effective batch 8 on one device. Local means [2,4,6,8] divided by four accumulate to 5 at unchanged parameters, followed by **one** optimizer update. Token-averaged losses require appropriate supervised-token weighting; stochastic and batch-dependent computation can differ from a literal large batch.
- Conventional full-context raw cache: `2 * L * B * T * H_kv * d * s` bytes. With `(L,B,T,H_kv,d,s)=(4,2,128,2,16,2)`, this is 131072 bytes = 128 KiB. Doubling T gives 256 KiB; doubling B too gives 512 KiB. KV heads can differ from query heads; metadata, static allocation, padding, sharing and specialized architectures need separate accounting.
- Zero-queue toy: one 20-token output in 1 s gives 20 generated tokens/s and 1 s completion. Four simultaneous 20-token outputs in 2 s give 40 generated tokens/s and 2 s completion each. This is not time to first token or a tail-latency benchmark.

**Constrained choice.** Require peak memory ≤10 GB, completion ≤1.5 s and a declared task score ≥0.85. Illustrative A=(12 GB,1.0 s,0.90) fails memory; B=(8 GB,2.0 s,0.88) fails latency; C=(9 GB,1.2 s,0.86) passes all three. These thresholds are teaching examples, not course grading rules.

Distillation, quantization and compatible model merging each need their own task-quality and target-runtime comparison. Merging does not inherently reduce model size. A model's stored footprint is not all peak runtime memory. The tutorial's eventual GPU measurements must be produced by an actual run; no timing or quality outcome is prefilled.

Primary references: [PyTorch AMP](https://docs.pytorch.org/docs/stable/amp.html), [activation checkpointing](https://docs.pytorch.org/docs/stable/checkpoint), [gradient accumulation](https://huggingface.co/docs/accelerate/usage_guides/gradient_accumulation), [causal caching](https://huggingface.co/docs/transformers/cache_explanation).


### Visual companion

Track weights, optimizer states, activations and KV cache separately. A choice that lowers memory can increase computation. Compare latency and throughput under the same workload rather than assuming lower precision is faster.

## Topic — Knowledge Distillation: Teachers, Students & Soft Targets

**Theory/Demo · 8 minutes**

Record the four-token CPU example; use the matching animation for the mechanism. The optional GPU extension is separate and defaults off. The teacher/student logits and `T=2.0` below are unchanged from the supplied source.

Generated teacher sequences support data distillation through SFT. Handwritten Module 3 template practice is not teacher-generated data. Probability distillation needs aligned prediction contexts and vocabulary meanings, teacher probability access and an explicit sequence mask/reduction. Equal argmax is insufficient; zero KL does not certify factual support.

The animation's interpolation is illustrative, not recorded training. Compare the original student, distilled student and actual teacher on content, format and target runtime. Fewer parameters do not prove proportional cost savings. Optional GKD differs from the simple temperature-scaled KL helper.


### Distillation signal and objective

1. **Data distillation:** teacher generates outputs; a student can learn these sequences with SFT. Save actual teacher identity, prompts, outputs and human review. Synthetic templates alone are not this process.
2. **Probability distillation:** compare aligned teacher and student probability distributions. The teacher supplies detached targets; student parameters receive gradients. Access to aligned probabilities is required; open weights are one way to obtain it.

Temperature softens scores without adding evidence. With the supplied logits, `T=2` yields teacher probabilities approximately `[.652140, .239909, .088258, .019693]`; the student is uniform. Teacher-to-student KL is `.47344944`, and the scaled loss is `1.89379776`. PyTorch takes student log probabilities first and teacher probabilities second. The conventional `T²` factor compensates temperature-related gradient scaling, not exact invariance across temperatures.

The helper below is deliberately restricted to independent rows of shape `[N,V]`. For sequences `[B,S,V]`, align causal positions, mask padding and unwanted supervision positions, and divide by the actual valid-token count. Blind `batchmean` on a sequence tensor divides by B, not by the number of valid positions.


In [ ]:
import math
import torch
import torch.nn.functional as F

def distill_loss(student_logits, teacher_logits, T=2.0):
    """Scaled KL(teacher || student), averaged over aligned independent [N,V] rows."""
    if isinstance(T, bool) or not math.isfinite(float(T)) or T <= 0:
        raise ValueError("T must be finite and positive")
    if student_logits.ndim != 2 or student_logits.shape != teacher_logits.shape:
        raise ValueError("Use matching [N,V] rows; sequence masking is a separate operation")
    if not all(student_logits.shape) or not torch.isfinite(student_logits).all() or not torch.isfinite(teacher_logits).all():
        raise ValueError("Logits must be nonempty and finite")
    return F.kl_div(F.log_softmax(student_logits / T, dim=-1),
                    F.softmax(teacher_logits.detach() / T, dim=-1),
                    reduction="batchmean") * T * T

teacher = torch.tensor([[5.0, 3.0, 1.0, -2.0]], dtype=torch.float64)
student = torch.tensor([[2.0, 2.0, 2.0, 2.0]], dtype=torch.float64, requires_grad=True)
loss = distill_loss(student, teacher)
print("teacher probabilities", F.softmax(teacher / 2, dim=-1))
print(f"loss (uniform student): {loss.item():.6f}")
print(f"loss (matched student): {distill_loss(teacher.clone(), teacher).item():.6f}")
assert math.isclose(loss.item(), 1.893797762861853, abs_tol=1e-7)
loss.backward()
print("student gradient", student.grad, "teacher gradient", teacher.grad)


### Distillation design extension

The executable CPU KL example above teaches soft targets. Multi-model GPU distillation is optional and is not part of the supported run path. Before implementing it, verify vocabulary/token-ID alignment, context handling, held-out evaluation and the exact trainer version. The project requires explaining distillation; it does not require running an experimental GKD trainer.


### Prerequisite teacher checkpoint

The distillation run needs a loaded teacher. The following original adapter-folding cell prepares it; the corresponding topic explains the distinction between this operation and combining independent adaptations.

In [ ]:
fold_plan = {"base_id": BASE_ID, "base_revision": BASE_REVISION,
    "adapter_files": ADAPTER_FILES, "split_sha256": stable_sha(split_record),
    "dtype": str(DTYPE), "operation": "LoRA folded into unquantized base"}
if FOLD_MODE == "create":
    if FOLDED_DIR.exists(): raise FileExistsError("Preserve existing folded artifact; use a new named directory")
    base = AutoModelForCausalLM.from_pretrained(BASE_ID, revision=BASE_REVISION,
        dtype=DTYPE, device_map={"": torch.cuda.current_device()})
    adapted = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
    analyst = adapted.merge_and_unload(safe_merge=True)
    analyst.save_pretrained(str(FOLDED_DIR), safe_serialization=True); tok.save_pretrained(str(FOLDED_DIR))
    write_json(FOLDED_DIR / "fold_manifest.json", fold_plan)
    del adapted, base
elif FOLD_MODE == "load":
    saved = json.loads((FOLDED_DIR / "fold_manifest.json").read_text())
    if saved != fold_plan: raise ValueError("Folded artifact provenance differs from the requested reference")
    analyst = AutoModelForCausalLM.from_pretrained(str(FOLDED_DIR), dtype=DTYPE,
        device_map={"": torch.cuda.current_device()})
else:
    raise ValueError("FOLD_MODE must be create or load")
analyst.eval()
print("Loaded folded half-precision checkpoint; quality equivalence remains to be tested")


The adapter was trained against a quantized base, while this cell folds it into half-precision weights. That changes the evaluated representation. Re-score the folded reference before comparing further compression; the effect is not assumed negligible. The current bootstrap adapter is practice data, not automatically your real Module 3 submission.


In [ ]:
import re, os

# test_rows: preserved Module3 regression rows; prior test exposure remains documented.
print(f"frozen test set: {len(test_rows)} rows 🔒")

REQUIRED = ["## ROOT CAUSE ANALYSIS", "**Product:**", "**Defect:**", "**Severity:**",
            "**Evidence:**", "**Probable cause:**", "**Recommended action:**"]

def format_ok(text):
    """Mechanical format only: all headers and one exact LOW/MEDIUM/HIGH severity line."""
    if not isinstance(text, str) or not all(h in text for h in REQUIRED):
        return False
    lines = re.findall(r"(?m)^\*\*Severity:\*\*[^\n]*$", text)
    return len(lines) == 1 and bool(re.fullmatch(
        r"\*\*Severity:\*\*[ \t]*(LOW|MEDIUM|HIGH)[ \t]*", lines[0]))

def chat(model, text, max_new_tokens=350):
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int) or max_new_tokens <= 0:
        raise ValueError("max_new_tokens must be a positive integer")
    model.eval()
    msgs = [{"role": "user", "content": text}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                     return_dict=True, return_tensors="pt").to(model.device)
    if inputs["input_ids"].shape[1] + max_new_tokens > 2048:
        raise ValueError("Declared 2048-token prompt/output budget exceeded")
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


### No additional distillation model is loaded

Continue to the trained Analyst compression experiment. The following reserved cells intentionally perform no optional model training.


In [ ]:
# Optional distillation design: use the KL exercise above. No additional model is trained by this cell.


In [ ]:
# Optional distillation design: use the KL exercise above. No additional model is trained by this cell.


In [ ]:
# Optional distillation design: use the KL exercise above. No additional model is trained by this cell.


### Evidence limits

The scalar KL exercise establishes the loss calculation only. It is not a trained student, a content-quality measurement or a compression result.


## Topic — Model Merging: Combining Adaptations

### Learning focus
Distinguish adapter folding from combining independent adaptations. The video uses a complete two-weight model: W0=(0,0), WA=(2,1), WB=(-2,3). Their midpoint is (0,2). Probes xA=(1,0) and xB=(0,1) require outputs 2 and 3; the midpoint produces 0 and 2. Correct arithmetic does not guarantee preserved behavior.

SLERP changes the interpolation path; TIES trims magnitudes, elects signs and averages aligned nonzero updates; DARE randomly drops and rescales updates. Compatibility and held-out task evidence still matter. A shared architecture does not establish row correspondence when token meanings differ. Merging can improve over parents in some settings; no improvement is guaranteed for this course model.


### Coordinate ledger and optional configuration
The CPU cell below independently checks the video's exact toy arithmetic. It does not merge Qwen checkpoints or measure task quality. The YAML is an explicitly incomplete practice configuration: its second entry is the unadapted base, whose task vector relative to itself is zero. It therefore does not demonstrate combining two trained specialists. Replace it with a separately trained, compatible adaptation before claiming that experiment.

Record both exact base revisions, architecture, token meanings, adapter scales, coefficient choices and final precision. Select settings on validation data, then evaluate required tasks and runtime separately. Shared-base adapter routing is a possible baseline; do not assume several full model copies are required. No CPU completion-time guarantee is given.


In [ ]:
import math, json
W_A, W_B = (2., 1.), (-2., 3.)
W_mix = tuple((a+b)/2 for a,b in zip(W_A,W_B))
assert W_mix == (0., 2.)
assert sum(w*x for w,x in zip(W_mix,(1,0))) == 0
assert sum(w*x for w,x in zip(W_mix,(0,1))) == 2
updates = [[.8,.1,-.6,.02],[.7,-.4,.3,.03],[-.2,-.5,.4,.01]]
trimmed=[]
for row in updates:
    keep=set(sorted(range(4),key=lambda j:abs(row[j]),reverse=True)[:2])
    trimmed.append([x if j in keep else 0. for j,x in enumerate(row)])
merged=[]
for col in zip(*trimmed):
    total=sum(col); sign=1 if total>0 else -1 if total<0 else 0
    aligned=[x for x in col if x*sign>0]
    merged.append(sum(aligned)/len(aligned) if aligned else 0.)
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(merged,[.75,-.45,-.6,0]))
assert math.isclose(.5*0+.5*(.2/(1-.5)),.2)
print({"midpoint":W_mix,"ties_trimmed":trimmed,"ties_result":merged,"dare_expected_entry":.2})

# Practice only: this base-model entry is NOT a second trained specialist.
merge_config = """
models:
  - model: ./analyst-merged
    parameters: {weight: 0.6, density: 0.6}
  - model: Qwen/Qwen2.5-1.5B-Instruct
    parameters: {weight: 0.4, density: 0.6}
merge_method: ties
base_model: Qwen/Qwen2.5-1.5B-Instruct
dtype: float16
"""
print(merge_config)
# After compatibility review and replacing the placeholder, save merge.yaml.
# Optional command: mergekit-yaml merge.yaml ./analyst-ties
# Do not bypass compatibility checks. Rescore the resulting candidate.


**The acceptance test is non-negotiable:** a merged model must *re-pass each parent's eval*
before it replaces anything. With our placeholder second parent, TIES here just scales the
Analyst's deltas — but the checking code is the same for a real merge:

```python
# merged = AutoModelForCausalLM.from_pretrained("./analyst-ties", torch_dtype=DTYPE, device_map="auto")
# print("merged format adherence:", rate([chat(merged, x) for x in test_inputs]))   # parent 1's eval
# print("merged summarization score:", ...)                                          # parent 2's eval
```

## Topic — Quantization Explained: Precision, Memory & Quality

**Theory · 6–8 minutes**

Explains how scaling and rounding represent weights with fewer bits and why quantization introduces error. Distinguishes post-training quantization from quantization-aware training and connects the quantized QLoRA base to Module 3. Introduces weight versus activation precision, calibration and the need to measure quality and latency on the target runtime.



### Quantization changes a representation
Quantization can reduce selected weight storage; its quality and speed effects are empirical. Use a declared method, exact artifact and target workload.

Uniform teaching example (not NF4): s=0.2, codes -7 through 7, nearest rounding, one unused four-bit code. Weights [-1.13,-0.34,0.26,1.31] become codes [-6,-2,1,7], reconstructing [-1.2,-0.4,0.2,1.4]. For x=[1,-1,2,0.5], outputs are 0.385 and 0.300. A separate weight 2 clips to 1.4, so its 0.6 error exceeds the 0.1 half-step bound, which requires no clipping.

The hypothetical 1.5B-weight payload falls from 3 GB at 16 bits to 0.75 GB at 4 bits. One 32-bit scale per 64 weights adds 0.09375 GB. With 2 GB of other memory, totals are 5 and 2.84375 GB. This is decimal-byte arithmetic, not a Qwen measurement or a claim about actual NF4 metadata.

PTQ converts a previously trained model. Calibration requirements depend on the method. QAT commonly simulates quantization effects during training before conversion. QLoRA instead trains adapters through a frozen quantized base, with separate compute and adapter dtypes; it is not generic full-model QAT.

| Family | What to check |
|---|---|
| bitsandbytes 4/8-bit | Quantized modules, codebook, scales, compute dtype and supported hardware |
| GPTQ or AWQ | Representative calibration, method settings, export and inference-engine compatibility |
| GGUF | Container tensor types, quantization variant and intended runtime; not one fixed bit width or CPU-only format |

Do not infer end-to-end GPU memory from get_memory_footprint alone. Measure allocated/reserved peak with declared batch, prompt and generated lengths. Quality, actual new-token throughput and latency require separate checks.


### Learning checkpoint
Name the stored quantity whenever stating a bit width. Weight payload, scale metadata, unquantized modules, activations, accumulators, cache and workspace are different terms. Smaller weight error can still change a greedy token at a narrow logit margin. The next tutorial measures the actual Analyst artifact rather than adopting the teaching ledger as a benchmark.

References: [Transformers bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes), [PyTorch QAT](https://pytorch.org/blog/quantization-aware-training/), [QLoRA](https://arxiv.org/abs/2305.14314). CPU arithmetic checked independently; GPU artifact conversion and performance rehearsal remain pending.


## Topic — Quantize the Analyst & Compare Its Footprint

**Tutorial: Colab · 6–8 minutes**
Show the Module 3 artifact, base revision and frozen row IDs. Load the original reference, the folded half-precision checkpoint and its load-time INT8 and NF4 candidates one at a time. Record actual API-reported model footprint and CUDA allocations; no fourfold total-memory reduction is assumed. Preserve the same inputs for the corresponding topic. Run the shared setup and folded-checkpoint prerequisite first; the optional distillation design is not required.


### Predeclare the practice comparison before outputs
The inherited five-point format and scaled-cosine thresholds are **practice checks**, not proof of factual correctness or a course-wide shipping policy. Declare project criteria separately before final evaluation. Format uses the Module 3 LOW/MEDIUM/HIGH contract. Save artifact hashes, input IDs, generation settings and thresholds before obtaining candidate outputs. Any changed plan needs a new named run folder. A failed candidate falls back only to an evaluated usable reference.

The two compressed candidates are folded INT8 and folded NF4. Folded half precision provides a footprint denominator and an evaluated fallback candidate. Declare the price source/date and intended arm before measurements.


In [ ]:
from huggingface_hub import model_info
EMBEDDING_REVISION=model_info("BAAI/bge-small-en-v1.5").sha
GPU_PRICE_PER_H = None  # Fill actual price before the comparison; record price source/date below.
PRICE_SOURCE = None
INTENDED_ARM = "folded_nf4"
GATE = {"max_format_drop_pts": 5.0, "max_content_drop_pts": 5.0}
COMPARISON_ROOT = Path("analyst_compression_v1")
GENERATION = {"max_new_tokens": 350, "do_sample": False, "context_budget": 2048}
comparison_plan = {"reference": "module3_reference", "candidates": ["folded_int8", "folded_nf4"], "half_precision_denominator": "folded_half",
    "base_id": BASE_ID, "base_revision": BASE_REVISION, "adapter_files": ADAPTER_FILES,
    "folded_files": artifact_files(FOLDED_DIR), "test_rows": test_rows,
    "gate": GATE, "gate_units":{"format":"percentage points", "content":"100 * raw cosine difference"},
    "hourly_price":GPU_PRICE_PER_H, "price_source":PRICE_SOURCE, "intended_arm":INTENDED_ARM, "generation": GENERATION, "dtype": str(DTYPE),
    "embedding_revision":EMBEDDING_REVISION,
    "runtime": {x: version(x) for x in ["torch","transformers","peft","bitsandbytes"]},
    "limits": "Previously exposed Module3 test rows are regression checks; cosine is not factual accuracy."}
COMPARISON_ROOT.mkdir(exist_ok=True)
plan_path = COMPARISON_ROOT / "plan.json"
if plan_path.exists():
    if json.loads(plan_path.read_text()) != comparison_plan: raise ValueError("Changed comparison plan: use a new named run folder")
else: write_json(plan_path, comparison_plan)
print("Predeclared plan", stable_sha(comparison_plan), GATE)


In [ ]:
def release_previous_models():
    for name in ["analyst", "analyst_4bit", "teacher", "student", "vanilla", "trained"]:
        globals().pop(name, None)
    gc.collect(); torch.cuda.empty_cache()

def load_arm(name):
    if name == "module3_reference":
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
        base_model = AutoModelForCausalLM.from_pretrained(BASE_ID, revision=BASE_REVISION,
            quantization_config=quant, device_map={"": torch.cuda.current_device()})
        base_model = prepare_model_for_kbit_training(base_model)
        result = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=False)
    elif name == "folded_half":
        result = AutoModelForCausalLM.from_pretrained(str(FOLDED_DIR), dtype=DTYPE,
            device_map={"": torch.cuda.current_device()})
    elif name == "folded_int8":
        result=AutoModelForCausalLM.from_pretrained(str(FOLDED_DIR),
            quantization_config=BitsAndBytesConfig(load_in_8bit=True),dtype=DTYPE,
            device_map={"":torch.cuda.current_device()})
    elif name == "folded_nf4":
        result = AutoModelForCausalLM.from_pretrained(str(FOLDED_DIR),
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=False, bnb_4bit_compute_dtype=DTYPE),
            device_map={"": torch.cuda.current_device()})
    else: raise ValueError("Unknown arm")
    result.eval(); result.config.use_cache = True
    return result

release_previous_models()
footprints = []
for name in ["module3_reference", "folded_half", "folded_int8", "folded_nf4"]:
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    loaded = None
    try:
        loaded = load_arm(name); torch.cuda.synchronize()
        record = {"arm": name, "model_footprint_bytes": int(loaded.get_memory_footprint()),
            "cuda_allocated_bytes_after_load": torch.cuda.memory_allocated(),
            "cuda_reserved_bytes_after_load": torch.cuda.memory_reserved(),
            "cuda_peak_allocated_bytes_during_load": torch.cuda.max_memory_allocated(),
            "parameter_dtypes": sorted({str(x.dtype) for x in loaded.parameters()}),
            "device": torch.cuda.get_device_name(), "status": "loaded"}
    except Exception as exc:
        record = {"arm": name, "status": "load_error", "error": repr(exc)}
    finally:
        del loaded; gc.collect(); torch.cuda.empty_cache()
    footprints.append(record)
    write_json(COMPARISON_ROOT / "footprints.json", {"plan_sha256": stable_sha(comparison_plan), "records": footprints})
    print(record)
print("These are load measurements. Generation cache and workspace need their own workload measurement.")

half=next((r["model_footprint_bytes"] for r in footprints if r["arm"]=="folded_half" and r["status"]=="loaded"),None)
for row in footprints:
    row["footprint_ratio_to_half"]=row.get("model_footprint_bytes",0)/half if half and row["status"]=="loaded" else None
write_json(COMPARISON_ROOT/"footprints.json",{"plan_sha256":stable_sha(comparison_plan),"records":footprints})


### Read the measured scope
`get_memory_footprint()` reports a model-level estimate, not complete process or peak-generation memory. CUDA allocated and reserved bytes describe different allocator quantities. A clean runtime, sequential loads and the same device reduce comparison ambiguity. Keep load failures and out-of-memory errors visible; they are not zero-byte successes.

No GPU run was executed during production. The copied half-precision checkpoint on disk remains half precision when NF4 is applied at load time. Do not claim a compressed on-disk artifact unless it is separately saved and reloaded successfully. GGUF export is an optional separate deployment path with its own supported conversion, quantization, runtime and quality checks; no fixed conversion time is promised.

References: https://huggingface.co/docs/transformers/quantization/bitsandbytes and https://huggingface.co/docs/peft/package_reference/peft_model.


## Topic — Apply the Quality Gate & Measure the Trade-Off

**Tutorial: Colab · 6–8 minutes**
Use the preceding topic’s saved plan and the same inputs for all four arms. Keep full outputs and generation failures. Read format adherence and the legacy embedding-cosine proxy separately; the proxy is not factual accuracy. Benchmark a declared workload with warmup, synchronized wall time and actual new token IDs. A passing pair of practice checks still requires human evidence review and workload acceptance. No GPU evaluation or benchmark was run during production.


### Compare against the intended reference
The frozen plan precedes candidate outputs. The code below evaluates `module3_reference`, `folded_half`, `folded_int8` and `folded_nf4` sequentially. All candidate transformations are compared with the original Module 3 reference; the folded pair can also be inspected to isolate load-time quantization.

A format drop is in percentage points. The content proxy is **mean normalized embedding cosine**, whose difference is multiplied by 100 for the inherited practice threshold; it is not a percentage of correct claims. Failed generation blocks the proxy gate rather than disappearing from its denominator. Review product identity, source support, severity, uncertainty and proposed action in the complete reports.

The benchmark uses the first three frozen prompts, ten cycles, two excluded warmups and a 150-new-token cap. Quality generation retains its separate 350-token cap. Timings include tokenization, transfer, generation and decoding, with CUDA synchronization at both boundaries; they exclude model load, retrieval, image inspection and queueing. Thirty observations give a small descriptive sample, not a production tail guarantee. Benchmark plan and device are saved before running.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

benchmark_plan = {"comparison_plan_sha256": stable_sha(comparison_plan),
    "prompt_ids": [r["row_id"] for r in test_rows[:3]], "cycles": 10, "warmups": 2,
    "max_new_tokens": 150, "batch_size": 1, "device": torch.cuda.get_device_name(),
    "timing_scope": "tokenize+transfer+generate+decode; CUDA synchronized; load and queue excluded"}
bench_plan_path = COMPARISON_ROOT / "benchmark_plan.json"
if bench_plan_path.exists():
    if json.loads(bench_plan_path.read_text()) != benchmark_plan: raise ValueError("Changed benchmark plan; use a new named run folder")
else: write_json(bench_plan_path, benchmark_plan)

def generate_record(model, text, max_new_tokens):
    if isinstance(max_new_tokens,bool) or not isinstance(max_new_tokens,int) or max_new_tokens < 1:
        raise ValueError("Positive integer token limit required")
    model.eval(); torch.cuda.synchronize(); start = time.perf_counter()
    inputs = tok.apply_chat_template([{"role":"user","content":text}], add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    prompt_tokens = int(inputs["input_ids"].shape[1])
    if prompt_tokens + max_new_tokens > GENERATION["context_budget"]:
        raise ValueError("Prompt plus output budget exceeds the declared context budget")
    with torch.inference_mode():
        sequence = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
            num_beams=1, use_cache=True, pad_token_id=tok.eos_token_id)
    new_ids = sequence[0,prompt_tokens:].detach().cpu().tolist()
    text_out = tok.decode(new_ids, skip_special_tokens=True).strip()
    torch.cuda.synchronize(); elapsed = time.perf_counter()-start
    return {"status":"ok", "text":text_out, "token_ids":new_ids, "new_tokens_including_eos":len(new_ids),
        "prompt_tokens":prompt_tokens, "seconds":elapsed, "reached_token_limit":len(new_ids)>=max_new_tokens}

def latency_summary(records):
    if not records or any(r.get("status") != "ok" for r in records):
        raise ValueError("Complete successful benchmark observations required; keep failures separately")
    seconds = np.asarray([r["seconds"] for r in records],dtype=float)
    counts = np.asarray([r["new_tokens_including_eos"] for r in records],dtype=float)
    if not np.isfinite(seconds).all() or (seconds<=0).any() or not np.isfinite(counts).all() or (counts<0).any():
        raise ValueError("Invalid benchmark values")
    return {"n":len(records), "mean_s":float(seconds.mean()),
        "p50_s":float(np.percentile(seconds,50)), "p95_s":float(np.percentile(seconds,95)),
        "generated_tokens_per_s":float(counts.sum()/seconds.sum()),
        "sequential_requests_per_s":float(len(records)/seconds.sum()),
        "percentile_method":"linear", "token_scope":"actual new IDs including EOS; no retokenization"}

release_previous_models()
arm_results = {}
for arm in ["module3_reference","folded_half","folded_int8","folded_nf4"]:
    result_path = COMPARISON_ROOT / f"{arm}_outputs.json"
    if result_path.exists():
        saved = json.loads(result_path.read_text())
        if saved["plan_sha256"] != stable_sha(comparison_plan) or saved["benchmark_plan_sha256"] != stable_sha(benchmark_plan):
            raise ValueError("Saved arm does not match the declared plans")
        if saved.get("complete"):
            arm_results[arm]=saved; continue
        raise ValueError("Partial run retained. Inspect its errors and use a new named run folder for a retry.")
    loaded = None
    result = {"arm":arm, "plan_sha256":stable_sha(comparison_plan),
        "benchmark_plan_sha256":stable_sha(benchmark_plan), "outputs":[], "benchmark":[], "complete":False}
    try:
        loaded = load_arm(arm)
        for row in test_rows:
            try: output = generate_record(loaded,row["input"],GENERATION["max_new_tokens"])
            except Exception as exc: output = {"status":"generation_error","text":"","error":repr(exc)}
            result["outputs"].append({"row_id":row["row_id"],"input":row["input"],"reference":row["output"],**output})
            write_json(result_path,result)
        prompts = test_rows[:3]
        for _ in range(benchmark_plan["warmups"]): generate_record(loaded,prompts[0]["input"],150)
        torch.cuda.reset_peak_memory_stats()
        for cycle in range(benchmark_plan["cycles"]):
            for row in prompts:
                result["benchmark"].append({"row_id":row["row_id"],"cycle":cycle,
                    **generate_record(loaded,row["input"],150)})
            write_json(result_path,result)
        result["benchmark_summary"] = latency_summary(result["benchmark"])
        result["peak_allocated_bytes_during_benchmark"] = torch.cuda.max_memory_allocated()
        result["complete"] = True
    except Exception as exc:
        result["run_error"] = repr(exc)
    finally:
        del loaded; gc.collect(); torch.cuda.empty_cache()
    write_json(result_path,result); arm_results[arm]=result

# CPU embedder prevents its parameters from contaminating GPU benchmark allocations.
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5",revision=EMBEDDING_REVISION,device="cpu")
reference_vectors = embedder.encode([r["output"] for r in test_rows],normalize_embeddings=True)

def score_arm(result):
    outputs=result["outputs"]
    ids=[x["row_id"] for x in outputs]
    expected=[x["row_id"] for x in test_rows]
    complete = ids==expected and all(x["status"]=="ok" for x in outputs)
    fmt=sum(format_ok(x["text"]) for x in outputs if x["status"]=="ok")/len(test_rows)
    proxy=None
    if complete:
        vectors=embedder.encode([x["text"] for x in outputs],normalize_embeddings=True)
        proxy=float((vectors*reference_vectors).sum(axis=1).mean())
        if not math.isfinite(proxy): proxy=None
    return {"format_rate":fmt,"mean_cosine_proxy":proxy,"complete_generation":complete,
        "expected_rows":len(test_rows),"returned_rows":len(outputs),
        "reached_limit":sum(bool(x.get("reached_token_limit")) for x in outputs)}

scores={name:score_arm(result) for name,result in arm_results.items()}
write_json(COMPARISON_ROOT / "quality_scores.json", {"plan_sha256":stable_sha(comparison_plan),"scores":scores,
    "proxy_encoder":"BAAI/bge-small-en-v1.5","proxy_encoder_revision":EMBEDDING_REVISION,"proxy_max_seq_length":embedder.max_seq_length,
    "limits":"Encoder may truncate long reports; cosine does not verify claims. Full human review required."})
for name,result in arm_results.items():
    print(name,scores[name])
    for output in result["outputs"]: print(output["row_id"],output["status"],"\n",output["text"])


In [ ]:
def practice_gate(reference, candidate, gate):
    if not reference["complete_generation"] or not candidate["complete_generation"]:
        return {"status":"INVALID INCOMPLETE GENERATION", "passes":False}
    a=reference["mean_cosine_proxy"]; b=candidate["mean_cosine_proxy"]
    if a is None or b is None or not all(math.isfinite(x) for x in [a,b]):
        return {"status":"INVALID PROXY", "passes":False}
    fmt_drop=100*(reference["format_rate"]-candidate["format_rate"])
    proxy_drop=100*(a-b)
    passed=fmt_drop<=gate["max_format_drop_pts"]+1e-9 and proxy_drop<=gate["max_content_drop_pts"]+1e-9
    return {"status":"PRACTICE CHECKS PASS" if passed else "REJECT CANDIDATE",
        "passes":passed,"format_drop_percentage_points":fmt_drop,"scaled_cosine_drop":proxy_drop}

gate_results={arm:practice_gate(scores["module3_reference"],scores[arm],GATE)
    for arm in ["folded_half","folded_int8","folded_nf4"]}
write_json(COMPARISON_ROOT / "gate_results.json", {"plan_sha256":stable_sha(comparison_plan),"results":gate_results})
for arm,result in gate_results.items(): print(arm,result)
review_path=COMPARISON_ROOT / "human_decision.json"
if not review_path.exists():
    write_json(review_path,{"plan_sha256":stable_sha(comparison_plan),"decision":"PENDING HUMAN REVIEW",
        "chosen_arm":None,"chosen_artifact":None,"claim_support_review":None,"severity_and_action_review":None,
        "runtime_acceptance":None,"fallback":None,"reason":None,
        "rule":"A failed or invalid candidate is not ready to ship. Retain only an evaluated usable reference; otherwise report no accepted artifact."})
print("Practice checks are not release approval. Inspect complete reports and declare an evidence-based decision.")


### Read the decision before the speed table
The two mechanical checks require both thresholds. A smaller or faster candidate does not override failure. A pass still leaves semantic review and operational requirements unresolved. Preserve the original reference if it meets the actual task requirements; do not invent an untested eight-bit fallback.

Inspect any output that reaches the token cap. The benchmark cap is a declared short workload; it is not automatically a complete report. Keep quality and throughput conclusions within the workload actually measured.


In [ ]:
for name,result in arm_results.items():
    print(name, result.get("benchmark_summary", {"status":"NO COMPLETE BENCHMARK","error":result.get("run_error")}))
    print("Peak benchmark allocated bytes",result.get("peak_allocated_bytes_during_benchmark"))
print("p50 and p95 are descriptive sample percentiles, not production latency guarantees.")
print("Generated-token throughput counts actual new IDs, including EOS; it excludes prompt tokens.")


### Speed is a measured outcome
Four-bit loading does not guarantee faster generation. Kernels, device, input/output lengths and batch size influence the result. This measurement has no serving queue or concurrency; p95 over a small repeated workload is not a service-level guarantee. Rehearse on the deployment runtime before making deployment claims.


In [ ]:
# Price was predeclared in comparison_plan; change the plan/run version before changing this assumption.

def sequential_cost_per_1000(mean_seconds, hourly_price):
    if hourly_price is None: return None
    if any(isinstance(x,bool) or not isinstance(x,(int,float)) or not math.isfinite(x) or x<0
           for x in [mean_seconds,hourly_price]): raise ValueError("Finite nonnegative time and price required")
    return mean_seconds*hourly_price/3600*1000

cost_rows=[]
for name,result in arm_results.items():
    stats=result.get("benchmark_summary")
    estimate=sequential_cost_per_1000(stats["mean_s"],GPU_PRICE_PER_H) if stats else None
    cost_rows.append({"arm":name,"hourly_price":GPU_PRICE_PER_H,"estimated_active_gpu_cost_per_1000":estimate,
        "scope":"mean sequential request time; excludes idle capacity, load, retrieval, storage and other services"})
    print(cost_rows[-1])
write_json(COMPARISON_ROOT / "cost_assumptions.json",cost_rows)
# Batching needs its own measured throughput and utilization; never divide cost blindly by batch size.


## Topic — Explaining the Module Project: Optimized Analyst

### Published competency evidence map

The supplied v2.0 rubric governs these criteria. The tutorial illustrates methods; the learner must supply their own actual project evidence. Review the must-pass criterion first.

| Criterion | Evidence | Met standard |
|---|---|---|
| C4.1 — Your optimization choice shows understanding | Technique note beside D4.1-D4.2; links to the selected model configurations. | Distinguish data from model parallelism, training from inference optimization, teacher-output from logit distillation, adapter folding from multi-model merging, and reduced-precision quantization. Identify the RiskLens bottleneck and justify the chosen route and trade-offs. Independent multi-GPU training, distillation and multi-parent merging are optional extensions, not required implementations. |
| C4.2 **Must-pass** — Your quality gate precedes the comparison | D4.1 GATE and pre-score snapshot/version; baseline and candidate inventory. | Before compressed-model re-scoring, record justified maximum format and content drops, their units, the Stage 3 reference, candidate arms, hourly price and intended ship arm. Preserve the dated version; keep thresholds unchanged for that comparison. Explain why the tolerances are acceptable for the stated use case rather than choosing them to guarantee a pass. |
| C4.3 — Your compression comparison uses the trained Analyst | D4.2 memory table; D4.3 saved generations, quality table and Module 4 checkpoint. | Produce at least two compression arms of the learner-trained Analyst and record memory footprints and ratios against half precision. Re-score each arm and the Stage 3 reference on the same frozen rows for format and content, with consistent settings. Show threshold comparisons and every verdict. Identify content cosine similarity as a proxy, not factual correctness. |
| C4.4 — Your serving estimates are measured and qualified | D4.4 benchmark outputs for reference and each arm; settings and cost calculation. | Report p50/p95 latency, actual generated-token throughput and estimated cost per 1,000 reports for the reference and each arm. State hardware, run count, warm-up, input/output lengths and hourly-price/utilization assumptions. Label small-sample percentiles and cost estimates appropriately; explain why lower precision may save memory without reducing latency. |
| C4.5 — Your deployment decision respects the gate | D4.3 decision note; accepted/rejected arm inventory; fallback and limits. | Select a candidate that passes the declared gate or reject failed candidates and document a feasible fallback. Link the decision to quality, memory and runtime evidence. An honestly evaluated failed compression can meet this competency; it cannot be labeled production-ready. Keep the original gate record and distinguish any later candidate or fallback evaluation. |

All criteria must be Met for this competency to be Mastered. See [practice-to-capstone differences](../../docs/ASSESSMENT.md) and the [exact consolidated rubric](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone/blob/codex/competency-colab-refresh/RUBRICS.md). Preserve full outputs and disclose limitations; no metric compensates for missing required evidence.

The lab includes two compressed candidates, folded INT8 and folded NF4, plus the original reference and a folded half-precision control. A rejected candidate can support an honest competency demonstration with a feasible fallback; it does not imply readiness to deploy.


### Worked example for the project explanation

**Illustrative arithmetic, not measured GPU results.** Forty fixed cases give reference format 36/40 = .90, A 35/40 = .875, and B 32/40 = .80. Format drops are 2.5 and 10 percentage points. Mean embedding cosine .88, .84 and .87 gives scaled drops 4 and 1. With inclusive practice limits of five on both measures, A passes the numerical gate; B fails format. A still needs claim-support and runtime review.

Candidate A's five sequential times [.6, .7, .8, .9, 1.0] seconds have mean and median .8 and linear sample p95 .98. Five requests producing 150 actual new token IDs each give 750 / 4 = 187.5 generated tokens/s. Count returned IDs, including EOS when returned. Record the timing scope, warmup exclusion, synchronization and raw observations.

At a hypothetical $0.90/hour, active sequential cost per thousand requests is mean_seconds × .90 / 3600 × 1000. Means 1.2, .8 and .6 give $0.30, $0.20 and $0.15. This excludes idle time, loading, retrieval and other services. Do not divide mechanically by batch size.

Model footprints 3.2, 1.3 and .9 GB are a separate illustrative ledger, not total request memory. Record cache, workspace and allocator measurements with clear boundaries. In the alternate content case, “UV damage caused the crack” is unsupported by “arrived cracked” and “polypropylene,” even if the cosine check passes.

**Handoff to Module 5:** selected reloadable artifact, exact revisions and configuration, full comparison evidence, cost assumptions, explicit decision and unresolved limitations. System integration requirements D4.5–D4.7 follow in Module 5.
